# 03 — Power `p` Sweep (PowerSoftMinLoss)

## Hypothesis

The exponent `p` in `PowerSoftMinLoss` raises the softmin weights to the power `p` before computing the weighted sum, sharpening the effective assignment distribution.

- **p = 1**: Equivalent to standard softmin / Chamfer-like weighting. Gradients are spread broadly across all slots.
- **p → ∞**: Approaches a hard nearest-neighbour assignment. Gradient is concentrated on the single closest slot, risking premature greediness and poor coverage of rare/distant objects.
- **Optimal p**: There is a sweet spot where sharpening improves slot specialisation without collapsing the gradient to a single winner.

**Experiment**: Fix τ = 0.12, sweep `p` ∈ {1.0, 1.5, 2.0, 3.0, 4.0, 5.0} and track convergence speed, final matched distance, slot diversity, and gradient diversity.

In [1]:
import sys
sys.path.insert(0, '.')
import clevr_utils as cu
from influencerformer.losses import PowerSoftMinLoss
print(f'Device: {cu.DEVICE}')

Device: cuda


In [2]:
P_LIST = [1.0, 1.5, 2.0, 3.0, 4.0, 5.0]
TAU = 0.12
N_EPOCHS = 50
SNAPSHOT_EPOCHS = {0, 5, 15, 35, 49}

In [ ]:
all_results = {}

for p in P_LIST:
    loss_fn = PowerSoftMinLoss(temperature=TAU, power=p)
    label = f'p={p}'
    print(f'\n=== Training {label} ===')
    metrics, snapshots = cu.train_and_monitor(
        loss_fn,
        loss_name=label,
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        tau_for_entropy=TAU,
    )
    all_results[label] = (metrics, snapshots)


=== Training p=1.0 ===


/global/homes/d/danieltm/.conda/envs/influencer/lib/python3.11/site-packages/torch/nn/modules/transformer.py:296: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343970094/work/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


  [p=1.0] epoch   0 | loss=3.4032 dist=1.6363 div=1.560 H=0.208 gd=0.739 lr=3.00e-04
  [p=1.0] epoch   1 | loss=2.9764 dist=1.5462 div=1.607 H=0.186 gd=0.637 lr=3.00e-04
  [p=1.0] epoch   2 | loss=2.7422 dist=1.4519 div=1.703 H=0.195 gd=0.723 lr=3.00e-04
  [p=1.0] epoch   3 | loss=2.4877 dist=1.3729 div=1.811 H=0.206 gd=0.679 lr=3.00e-04
  [p=1.0] epoch   4 | loss=2.2483 dist=1.2926 div=1.884 H=0.201 gd=0.851 lr=3.00e-04
  [p=1.0] epoch   5 | loss=2.0574 dist=1.2189 div=1.954 H=0.199 gd=0.833 lr=3.00e-04
  [p=1.0] epoch   6 | loss=1.9129 dist=1.1721 div=2.021 H=0.194 gd=0.739 lr=3.00e-04
  [p=1.0] epoch   7 | loss=1.7977 dist=1.1389 div=2.039 H=0.198 gd=0.791 lr=3.00e-04
  [p=1.0] epoch   8 | loss=1.7238 dist=1.1187 div=2.047 H=0.206 gd=0.798 lr=3.00e-04
  [p=1.0] epoch   9 | loss=1.6606 dist=1.1022 div=2.051 H=0.215 gd=0.821 lr=3.00e-04
  [p=1.0] epoch  10 | loss=1.6159 dist=1.0826 div=2.049 H=0.219 gd=0.782 lr=3.00e-04
  [p=1.0] epoch  11 | loss=1.5773 dist=1.0744 div=2.075 H=0.221 g

In [ ]:
cu.plot_monitoring(all_results, title='Power sweep — τ=0.12')

best_p = min(all_results, key=lambda k: all_results[k][0]['val_matched_dist'][-1])
worst_p = max(all_results, key=lambda k: all_results[k][0]['val_matched_dist'][-1])
print(f'Best p by final val dist:  {best_p}')
print(f'Worst p by final val dist: {worst_p}')

cu.plot_pca_snapshots(all_results[best_p][1],  title=f'PCA Snapshots — best ({best_p})')
cu.plot_pca_snapshots(all_results[worst_p][1], title=f'PCA Snapshots — worst ({worst_p})')

In [ ]:
cu.summary_table(all_results)